In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
# create two related test dataframes
persons = session.create_dataframe([
    [1, "John", 2], [2, "Mary", None], [3, "Mark", 2]],
    schema=["id", "name", "id_parent"])
persons

data = session.create_dataframe([
    [1, "Teacher", 28], [2, "Engineer", 22], [4, "Architect", 45]],
    schema=["id", "profession", "age"])
data

In [ ]:
# if joins columns have the same names (def INNER join)
persons.join(data, "id")

In [ ]:
# on columns with the same names
persons.natural_join(data)

In [ ]:
# w/ join conditiion (auto-prefix on joined columns, if the same name)
#persons.join(data,
 #   persons.col("id_parent") == data.col("id"))


#
persons.join(data,
    persons.col("id") == data.col("id"))


In [ ]:
# renaming the columns in SELECT
persons.join(data,
    persons.col("id") == data.col("id")
    ).select(persons["id"].alias("id_person"), "name",
             data["id"].alias("id_data"), "profession", "age")

In [ ]:
# overriding directly the auto-prefix
persons.join(data,
    persons.col("id") == data.col("id"),
    lsuffix="_person", rsuffix="_data")

In [ ]:
# left outer join
persons.join(data,
    persons.col("id") == data.col("id"),
    how="left")

In [ ]:
# left anti join
persons.join(data,
    persons.col("id") == data.col("id"),
    how="leftanti")

In [ ]:
# full outer join
persons.join(data,
    persons.col("id") == data.col("id"),
    how="full")

In [ ]:
# ~persons.cross_join(data)
persons.join(data, how="cross")

In [ ]:
# self join (this will fail)
persons.join(persons,
    persons["id"] == persons["id_parent"])

In [ ]:
# fix for self-join
from copy import copy
parents = copy(persons)

parents.join(persons,
    persons["id_parent"] == parents["id"]
    ).select(
        persons["name"].alias("child"),
        parents["name"].alias("parent"))

In [ ]:
# other two test data frames for set operations
employees = session.create_dataframe([
    ["John", 28], ["Mary", 22], ["Mark", 51]],
    schema=["name", "age"])
employees

customers = session.create_dataframe([
    ["John", 28], ["Adele", 18], ["George", 34], ["Mark", 51]],
    schema=["full_name", "customer_age"])
customers

In [ ]:
employees.union(customers)

In [ ]:
employees.union_all(customers)

In [ ]:
# union by name will match sets by their column names
customers2 = customers.select(
    customers["customer_age"].alias("age"),
    customers["full_name"].alias("name"))
employees.union_by_name(customers2)   # here column name should be same

In [ ]:
employees.intersect(customers)

In [ ]:
# ~minus/except_
employees.subtract(customers)


In [ ]:
employees.select(employees.name).distinct().subtract(customers.select(customers.full_name).distinct())